# Revision de APRX con arcpy.mp

Este notebook lista los mapas de un proyecto APRX, permite escoger uno y extrae las rutas/fuentes de datos de todos sus layers.

In [1]:
from pathlib import Path

import arcpy
import pandas as pd

aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO OFFLINE.aprx"

if not Path(aprx_path).exists():
    raise FileNotFoundError(f"No existe el APRX: {aprx_path}")

project = arcpy.mp.ArcGISProject(aprx_path)
project

## 1. Listar mapas del APRX

In [2]:
maps = project.listMaps()

if not maps:
    raise RuntimeError("El APRX no contiene mapas.")

maps_df = pd.DataFrame(
    [{"index": index, "map_name": map_obj.name, "layer_count": len(map_obj.listLayers())} for index, map_obj in enumerate(maps)]
)
maps_df

,index,map_name,layer_count
0,0,MLP SIG PAO INGENIERIA FLUOR,83
1,1,MLP SIG PAO RIESGOS GEOTECNICOS OFFLINE,242
2,2,MLP SIG PAO MEDIDAS OFFLINE,421
3,3,Orto Transversal,5
4,4,MLP SIG PAO MEDIDAS,147
5,5,MLP SIG PAO LAYOUT OFFLINE v4,564
6,6,MLP SIG PAO LAYOUT OFFLINE v5,562
7,7,MLP SIG PAO LAYOUT OFFLINE v6,601


In [5]:
maps_df[maps_df['map_name'] == 'MLP SIG PAO LAYOUT OFFLINE v4']

,index,map_name,layer_count
5,5,MLP SIG PAO LAYOUT OFFLINE v4,564


## 2. Escoger mapa

In [6]:
# Cambia este indice usando la tabla anterior.
selected_map_index = 5

if selected_map_index < 0 or selected_map_index >= len(maps):
    raise IndexError(f"Indice de mapa invalido: {selected_map_index}. Usa un valor entre 0 y {len(maps) - 1}.")

selected_map = maps[selected_map_index]
print(f"Mapa seleccionado: {selected_map.name}")

Mapa seleccionado: MLP SIG PAO LAYOUT OFFLINE v4


## 3. Extraer rutas de layers

In [7]:
def safe_layer_value(layer, attr_name, default=None):
    try:
        return getattr(layer, attr_name)
    except Exception:
        return default


def get_layer_data_source(layer):
    try:
        if layer.supports("DATASOURCE"):
            return layer.dataSource
    except Exception:
        pass
    return None


def get_connection_info(layer):
    try:
        return layer.connectionProperties
    except Exception:
        return None


def iter_layers_recursive(container, parent_path="", depth=0, seen=None):
    if seen is None:
        seen = set()

    for layer in container.listLayers():
        layer_name = safe_layer_value(layer, "name", "")
        long_name = safe_layer_value(layer, "longName", layer_name)
        layer_path = long_name or (f"{parent_path}\\{layer_name}" if parent_path else layer_name)
        layer_key = layer_path or id(layer)

        if layer_key in seen:
            continue

        seen.add(layer_key)
        yield layer, layer_path, parent_path, depth

        if safe_layer_value(layer, "isGroupLayer", False):
            yield from iter_layers_recursive(layer, layer_path, depth + 1, seen)


layer_rows = []

for order, (layer, layer_path, parent_path, depth) in enumerate(iter_layers_recursive(selected_map), start=1):
    connection_info = get_connection_info(layer)
    layer_rows.append(
        {
            "order": order,
            "map_name": selected_map.name,
            "depth": depth,
            "group_path": parent_path,
            "layer_path": layer_path,
            "layer_name": safe_layer_value(layer, "name"),
            "long_name": safe_layer_value(layer, "longName"),
            "is_group_layer": safe_layer_value(layer, "isGroupLayer", False),
            "is_broken": safe_layer_value(layer, "isBroken", None),
            "visible": safe_layer_value(layer, "visible", None),
            "data_source": get_layer_data_source(layer),
            "connection_info": connection_info,
        }
    )

layers_df = pd.DataFrame(layer_rows)
layers_df.head()

,order,map_name,depth,group_path,layer_path,layer_name,long_name,is_group_layer,is_broken,visible,data_source,connection_info
0,1,MLP SIG PAO LAYOUT OFFLINE v4,0,,SIAS - PAO,SIAS - PAO,SIAS - PAO,False,False,True,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\...,"{'dataset': 'CL_MLP_PAO_SIAS_PO', 'workspace_f..."
1,2,MLP SIG PAO LAYOUT OFFLINE v4,0,,Cruces,Cruces,Cruces,True,False,False,None,None
2,3,MLP SIG PAO LAYOUT OFFLINE v4,1,Cruces,Cruces\Atraviesos Viales FLUOR,Atraviesos Viales FLUOR,Cruces\Atraviesos Viales FLUOR,False,False,True,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\...,"{'dataset': 'CL_MLP_PAO_Atraviesos_FLUOR_PU', ..."
3,4,MLP SIG PAO LAYOUT OFFLINE v4,1,Cruces,Cruces\Atraviesos Viales Paralelismos,Atraviesos Viales Paralelismos,Cruces\Atraviesos Viales Paralelismos,False,False,True,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\...,"{'dataset': 'CL_MLP_PAO_Atraviesos_FLUOR_PO', ..."
4,5,MLP SIG PAO LAYOUT OFFLINE v4,1,Cruces,Cruces\Cruces Ferronor,Cruces Ferronor,Cruces\Cruces Ferronor,False,False,True,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\...,{'dataset': 'CL_MLP_PAO_OBRAS_PUNTUALES_EIA_PU...


In [8]:
import re
[c for c in layers_df['layer_path'] if re.search(r'.tpk|.tpkx',c,re.IGNORECASE)]

[]

## 4. Ver solo layers con ruta/fuente de datos

In [ ]:
layers_with_sources_df = layers_df[layers_df["data_source"].notna()].copy()
layers_with_sources_df[["order", "depth", "group_path", "layer_path", "is_broken", "data_source"]]

## 5. Exportar resultado a CSV

In [ ]:
safe_map_name = "".join(char if char.isalnum() or char in "-_" else "_" for char in selected_map.name)
output_csv = Path.cwd() / f"layers_{safe_map_name}.csv"
layers_with_sources_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
output_csv